# Upload trained weights + model cards to the Hugging Face Hub

Pushes your two trained checkpoints (on Google Drive) and their model cards to public HF repos so `texture_frames_de.FrameParser()` can download them. Run once, after training.

**Publishes model weights derived from SALSA/TIGER (academic/non-commercial).** Ensure this is consistent with the corpus licence before running (see the repo README's licence section).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q -U "huggingface_hub>=0.20"
# clone the repo to get the model cards (and the package, for the verify step)
!rm -rf /content/texture-frames-de
!git clone -q https://github.com/texturejc/texture-frames-de /content/texture-frames-de

In [ ]:
from huggingface_hub import login
login()   # paste a HF token with *write* access (hf.co/settings/tokens)

In [ ]:
import os
USER = "texturejc"
JOBS = [
    {"repo": f"{USER}/texture-frames-de-frame",
     "dir":  "/content/drive/MyDrive/Texture_Frames/models/frame2_de",
     "card": "/content/texture-frames-de/model_cards/frame/README.md",
     "need": ["frame2_model.pt", "frame2id.json"]},
    {"repo": f"{USER}/texture-frames-de-args",
     "dir":  "/content/drive/MyDrive/Texture_Frames/models/args2_de",
     "card": "/content/texture-frames-de/model_cards/args/README.md",
     "need": ["args2_model.pt", "role2id.json"]},
]
for j in JOBS:
    for f in j["need"]:
        assert os.path.exists(os.path.join(j["dir"], f)), f"missing {f} in {j['dir']}"
    assert os.path.exists(j["card"]), f"missing card {j['card']}"
print("checkpoints + cards present.")

## Push both repos (weights + tokenizer + model card)

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_file
api = HfApi()

for j in JOBS:
    create_repo(j["repo"], repo_type="model", exist_ok=True, private=False)
    api.upload_folder(repo_id=j["repo"], folder_path=j["dir"], repo_type="model")
    upload_file(path_or_fileobj=j["card"], path_in_repo="README.md",
                repo_id=j["repo"], repo_type="model")   # model card
    print("pushed ->", f"https://huggingface.co/{j['repo']}")

## Verify it loads (downloads from the Hub you just pushed)

In [ ]:
!pip install -q "transformers==4.57.6" sentencepiece simplemma
import sys; sys.path.insert(0, "/content/texture-frames-de/src")
from texture_frames_de import FrameParser
parser = FrameParser()
print(parser.parse("Die Polizei verhaftete den Verdächtigen ."))